In [1]:
from pymol import cmd, cgo
import sys
import pymol
import pickle
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as colors
from pathlib import Path
sys.path.append('/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM')

from src.visualize import (load_structure,
                           initialize_representation,
                           color_chain,
                           set_chain_representation,
                           draw_vector,
                           draw_sphere)

from src.preprocess import (extract_chains_chothia,
                            fix_case_if_reused)

# =====================================
# 1. Load your data
# =====================================

voxel_centers = voxels = np.load("voxel_centers.npy")

with open("my_first_carboxamid_map.plk","rb") as f:
    values = pickle.load(f)

pdbpath = "/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data/5ggs.pdb"

In [2]:
voxel_centers.shape
values.shape

(54740,)

In [2]:
#Quick mafs

#values = values.T

In [3]:
#extract chains
try:
    chain_pairs = extract_chains_chothia(pdbpath)
    if not chain_pairs:
        raise ValueError(f"No chain pairs returned for {pdbpath}")
    ligand_chains, receptor_chains = chain_pairs[0]
    receptor_chains = fix_case_if_reused(ligand_chains, receptor_chains, Path(pdbpath).stem) # some chothia pdbs have wrong headers
except Exception as e:
    print(f"[ERROR] Problem processing file: {pdbpath}")
    print(f"Reason: {e}")
    exit(1)  # Non-zero exit code so subprocess.run will detect failure



pymol.finish_launching()
obj_name = load_structure(pdbpath, object_name="myProtein") #load pdb
initialize_representation(object_name=obj_name)

cmd.hide("everything", obj_name)

threshold = 0.90

#ligand
c = "red"
repr = "lines"
for chain in ligand_chains:
    #color_chain(chain, c, object_name=obj_name)
    set_chain_representation(chain, repr, object_name=obj_name)


##receptor
#repr = "sticks"
#for chain in receptor_chains:
#    #color_chain(chain, c, object_name=obj_name)
#    set_chain_representation(chain, repr, object_name=obj_name)
#    selection = f"/{obj_name}//{chain}"
#    cmd.util.cbac(selection)

cmd.select("highlighted_residues", f"{obj_name} and resn GLN+ASN")
cmd.show("sticks", "highlighted_residues")
cmd.color("magenta", "highlighted_residues")

cmd.bg_color("white")

cmap = cm.get_cmap("viridis")
#norm = colors.Normalize(vmin=values.min(), vmax=values.max())
norm = colors.Normalize(threshold, vmax=values.max())

def value_to_rgb(val):
    r, g, b, _ = cmap(norm(val))
    return (r, g, b)

for i, (center, val) in enumerate(zip(voxel_centers, values)):
    if val >= threshold:
        color = value_to_rgb(val)
        name = f"sphere_{i}"
        draw_sphere(center, radius=0.5, color=color, alpha=0.8, name=name)

cmd.zoom()


 PyMOL(TM) Molecular Graphics System, Version 3.1.0.
 Copyright (c) Schrodinger, LLC.
 All Rights Reserved.
 
    Created by Warren L. DeLano, Ph.D. 
 
    PyMOL is user-supported open-source software.  Although some versions
    are freely available, PyMOL is not in the public domain.
 
    If PyMOL is helpful in your work or study, then please volunteer 
    support for our ongoing efforts to create open and affordable scientific
    software by purchasing a PyMOL Maintenance and/or Support subscription.

    More information can be found at "http://www.pymol.org".
 
    Enter "help" for a list of commands.
    Enter "help <command-name>" for information on a specific command.

 Hit ESC anytime to toggle between text and graphics.

/tmp/ipykernel_16297/1192804390.py:45: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` inst